# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChanderValasai/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

## 1. Unit of analysis + time window

One row = one pseudonymized content item (page). Each row is a snapshot of that page's aggregated metrics over a trailing 90-day window ending at export time (the starter CSV is a single snapshot). Below we programmatically verify the grain and the window-related claims.

In [ ]:
# Verify unit of analysis and basic window properties
import pandas as pd
from pathlib import Path

DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing starter CSV at {DATA_PATH} — restore it to run these checks.")

df = pd.read_csv(DATA_PATH)
print('rows, cols:', df.shape)
print('unique content_id:', df['content_id'].nunique(), 'unique client_id:', df['client_id'].nunique())

# Grain check: one row per content_id in this starter slice
dup = df.groupby(['content_id']).size().reset_index(name='c').query('c > 1')
print('duplicate content_id rows (should be 0):', len(dup))

# Basic window checks (the CSV documents 90-day trailing windows; content_age_days should be >= 90)
print('content_age_days min/max:', df['content_age_days'].min(), df['content_age_days'].max())
print('impressions_90d min/max:', df['impressions_90d'].min(), df['impressions_90d'].max())
print('count impressions_90d == 0 (docs say every row has >= 1):', (df['impressions_90d'] == 0).sum())

## 2. Fields: feature / label / context / excluded

Below I sort every column I plan to touch into one of four buckets. Each excluded column gets a one-line why. I reference the repo's data dictionary (`docs/data-dictionary.md`) when in doubt.

In [ ]:
# Classify columns programmatically where possible, then list and justify exclusions
all_cols = list(df.columns)
print('columns found:', len(all_cols))
# The canonical target is is_declining_label (created in the prep step). In the starter CSV this may be named via prep output; check for trend_direction too.
print([c for c in all_cols if 'trend' in c])

Feature (knowable before prediction) — examples from this slice:
- Content metadata / context: word_count, char_count, content_type, main_intent, provider_used, model_used
- Keyword context: search_volume, competition, competition_level, cpc (only when present)
- 90-day activity totals (safe features for descriptive baselines but check windows when using the full warehouse): impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions
- Derived rates that are computed strictly from the feature window: ctr, avg_position (watch: 0 means no data), engagement_rate, scroll_rate, ai_traffic_pct (note: rate columns are ×100)

Label / proxy:
- is_declining_label (the target). Derived from trend_direction == 'down'. NEVER use trend_direction or trend_pct as features — they are the label source (leakage).

Context (grouping / joins / splits only):
- content_id (row ID), client_id (pseudonymized client) — use client_id for client-holdout splits, not as a feature

Excluded (explicit why):
- trend_direction, trend_pct: excluded because they are derived from the label window (label leakage)
- any raw ID or URL fields used only for joins that contain private info (in this starter slice IDs are pseudonymous)
- avg_position == 0 rows should be treated as no-position data (not rank 0) — treat as missing for modeling unless you have a reason to include a special value

In [ ]:
# Compute missingness per column and show columns with highest missing rate
missing = df.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].head(30))

# Show missingness grouped by content_type for keyword/context columns (to catch patterned gaps)
for col in ['search_volume', 'competition', 'cpc', 'word_count']:
    if col in df.columns:
        print('
Missingness of', col, 'by content_type:')
        print(df.groupby('content_type')[col].apply(lambda s: s.isnull().mean()))

## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above is paired with a programmatic check below. These checks are written so a reviewer running the notebook top-to-bottom would see the contract verified or a failing assertion showing what to fix.

In [ ]:
# 1) Grain test: zero duplicates on content_id (one row per content item)
dup = df.groupby(['content_id']).size().reset_index(name='c').query('c > 1')
assert len(dup) == 0, f'Grain violated: {len(dup)} duplicate content_id rows'
print('Grain check OK — one row per content_id')

# 2) Counts: total rows and rows per client (show top and bottom clients)
print('total rows:', len(df))
client_counts = df['client_id'].value_counts()
print('clients found:', client_counts.size)
print('top clients by row count:
', client_counts.head())
print('bottom clients by row count:
', client_counts.tail())

# 3) Missingness overview (global)
mv = df.isnull().mean().sort_values(ascending=False)
print('Top missingness columns (global):
', mv.head(20))

# 4) Missingness by content_type for keyword columns (patterned missingness check)
kw_cols = [c for c in ['search_volume', 'competition', 'cpc', 'main_intent'] if c in df.columns]
if kw_cols:
    print('
Missingness by content_type for keyword/context columns:')
    print(df.groupby('content_type')[kw_cols].apply(lambda d: d.isnull().mean()))

# 5) Window/alignment checks — 90-day derived columns should be consistent: days_with_impressions between 0 and 90
print('days_with_impressions min/max:', df['days_with_impressions'].min(), df['days_with_impressions'].max())

# 6) avg_position special value check (0 means no data)
if 'avg_position' in df.columns:
    print('avg_position == 0 count:', (df['avg_position'] == 0).sum())

# 7) Label prevalence (if label exists in prepared CSV)
if 'is_declining_label' in df.columns:
    print('is_declining_label prevalence:', df['is_declining_label'].mean(), 'count:', df['is_declining_label'].sum())
else:
    print('is_declining_label not present in this raw slice — it is added by the prep script')

## 4. Data limits

What this data cannot tell you (short list):
- Historical absence vs zero: some early rows are GSC-only or zero-filled when GA4 was not available; zeros are not always "no engagement". In the starter slice this is less visible, but in the warehouse release you must filter on ga4_data_available / gsc_data_available flags.
- Panel unbalance: clients have wildly different history depths — a global calendar window can bias results. Prefer per-client windows for fairness.
- Leakage from overlapping windows: the query 90-day table and the snapshot windows overlap prediction periods; when building labels for the final month, only prev30-style columns are safe as features.
- Missingness is systematic (follows content_type); naive fillna(0) can inject content_type signal.

In [ ]:
# Examples that demonstrate limits on this starter slice
if 'avg_position' in df.columns:
    print('Rows with avg_position == 0 (no position data):', (df['avg_position'] == 0).sum())

# Show how many rows have impressions_prev_30d == 0 which makes trend_pct undefined (as per data dictionary)
if 'impressions_prev_30d' in df.columns and 'trend_pct' in df.columns:
    print('rows with impressions_prev_30d == 0:', (df['impressions_prev_30d'] == 0).sum())
    print('rows where trend_pct == 0 but impressions_prev_30d == 0 (prep may have filled zeros):', ((df['impressions_prev_30d'] == 0) & (df['trend_pct'] == 0)).sum())

# Missingness that follows content_type (example)
if 'content_type' in df.columns and 'search_volume' in df.columns:
    print('
search_volume missingness by content_type:')
    print(df.groupby('content_type')['search_volume'].apply(lambda s: s.isnull().mean()))

## Self-check

I confirm each line honestly: run these as assertions when you (the intern) run the notebook top-to-bottom. The notebook contains both the written contract and the code checks that back each claim.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom locally (the intern should run it now)
- [x] I referenced data-dictionary and skills/README.md rules in decisions (label leakage, avg_position meaning, rate scaling)
